<a href="https://colab.research.google.com/github/Kuzay3t/ViT-NAS-Compression/blob/master/Implementing_VITS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 (NEW - Simplified for Colab)
!git clone https://github.com/Kuzay3t/ViT-NAS-Compression.git
%cd ViT-NAS-Compression

# Install core dependencies only (skip setup.py)
!pip install -q torch torchvision transformers timm numpy pyyaml omegaconf tqdm tensorboard wandb onnx onnxruntime psutil

print("✅ Dependencies installed!")

In [ ]:
# Cell 2
from src.utils.device_info import DeviceInfo
from src.search_space.search_space import SearchSpace

print("✅ All imports successful!")

In [ ]:
# Cell 3
DeviceInfo.print_system_info()

In [ ]:
# Cell 4
search_space = SearchSpace("config/search_space.yaml")
search_space.print_search_space_info()

In [ ]:
# Cell 5 (Debug version)
from src.search_space.search_space import SearchSpace

search_space = SearchSpace("config/search_space.yaml")

# Generate a config
config = search_space.random_sample()
is_valid, errors = search_space.validate_config(config)

print("="*70)
print("CONFIG VALIDATION DEBUG")
print("="*70)

print(f"\n✓ Config generated successfully")
print(f"\nArchitecture:")
print(f"  - depth: {config.architecture.depth}")
print(f"  - embed_dim: {config.architecture.embed_dim}")
print(f"  - num_heads: {config.architecture.num_heads}")

print(f"\nCompression:")
print(f"  - pruning enabled: {config.compression.pruning.enabled}")
print(f"  - quantization enabled: {config.compression.quantization.enabled}")
print(f"  - pruning ratios: {config.compression.pruning.layer_wise_ratios[:3]}...")
print(f"  - quant bits: {config.compression.quantization.layer_wise_bits[:3]}...")

print(f"\nValidation Result: {is_valid}")

if not is_valid:
    print(f"\n❌ ERRORS FOUND:")
    for i, error in enumerate(errors, 1):
        print(f"  {i}. {error}")
else:
    print(f"\n✅ Config is VALID!")

print("="*70)

In [ ]:
# Cell 6: Create ViT Supernet files
import os

# Create directory
os.makedirs('src/supernet', exist_ok=True)

# Create __init__.py
with open('src/supernet/__init__.py', 'w') as f:
    f.write('# Supernet modules\n')

print("✓ Created src/supernet/__init__.py")

# Create vit_supernet.py
vit_code = '''import torch
import torch.nn as nn
from typing import Dict, Any
import math

class PatchEmbedding(nn.Module):
    """Convert image to patch embeddings."""

    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)  # (B, embed_dim, num_patches_h, num_patches_w)
        x = x.flatten(2)   # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (B, num_patches, embed_dim)
        return x


class Attention(nn.Module):
    """Multi-head self-attention."""

    def __init__(self, dim, num_heads=12, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class MLP(nn.Module):
    """MLP as used in Vision Transformer."""

    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class TransformerBlock(nn.Module):
    """Transformer encoder block."""

    def __init__(self, dim, num_heads, mlp_ratio=4., drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads, attn_drop=attn_drop)
        self.norm2 = nn.LayerNorm(dim)

        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(in_features=dim, hidden_features=mlp_hidden_dim, drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViTSupernet(nn.Module):
    """
    Vision Transformer Supernet supporting multiple architectures.
    """

    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 num_classes=1000, max_depth=16, max_embed_dim=768):
        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.max_depth = max_depth
        self.max_embed_dim = max_embed_dim

        # Patch embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, max_embed_dim)
        num_patches = self.patch_embed.num_patches

        # Class token and position embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, max_embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, max_embed_dim))
        self.pos_drop = nn.Dropout(p=0.1)

        # Transformer blocks (all possible blocks)
        self.blocks = nn.ModuleList([
            TransformerBlock(max_embed_dim, num_heads=12, mlp_ratio=4.0)
            for _ in range(max_depth)
        ])

        # Layer norm
        self.norm = nn.LayerNorm(max_embed_dim)

        # Classification head
        self.head = nn.Linear(max_embed_dim, num_classes)

        # Initialize weights
        nn.init.trunc_normal_(self.pos_embed, std=.02)
        nn.init.trunc_normal_(self.cls_token, std=.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x, config=None):
        """
        Forward pass with optional config for subnet selection.

        Args:
            x: Input image tensor (B, C, H, W)
            config: SearchConfig specifying which layers/dims to use

        Returns:
            Classification logits (B, num_classes)
        """
        B = x.shape[0]

        # Patch embedding
        x = self.patch_embed(x)  # (B, num_patches, embed_dim)

        # Add class token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        # Add position embeddings
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # Determine number of blocks to use
        if config is not None:
            num_blocks = config.architecture.depth
        else:
            num_blocks = self.max_depth

        # Apply transformer blocks
        for i in range(num_blocks):
            x = self.blocks[i](x)

        # Layer norm
        x = self.norm(x)

        # Classification (use class token)
        x = x[:, 0]
        x = self.head(x)

        return x
'''

with open('src/supernet/vit_supernet.py', 'w') as f:
    f.write(vit_code)

print("✓ Created src/supernet/vit_supernet.py")
print("\n✅ ViT Supernet files created!")

In [ ]:
# Cell 7: Test ViT Supernet
import torch
import sys

from src.supernet.vit_supernet import ViTSupernet
from src.search_space.search_space import SearchSpace

print("="*70)
print("ViT SUPERNET TEST")
print("="*70)

# Create supernet
model = ViTSupernet(img_size=224, patch_size=16, num_classes=1000)
print(f"\n✓ ViT Supernet created")

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"✓ Total parameters: {num_params:,}")

# Test forward pass with dummy input
x = torch.randn(2, 3, 224, 224)
print(f"✓ Input shape: {x.shape}")

output = model(x)
print(f"✓ Output shape: {output.shape}")

# Load search space
search_space = SearchSpace("config/search_space.yaml")
config = search_space.random_sample()

# Forward pass with config
output_with_config = model(x, config=config)
print(f"✓ Output with config (depth={config.architecture.depth}): {output_with_config.shape}")

print(f"\n✅ ViT Supernet Test PASSED!")
print("="*70)


In [ ]:
# Cell 9 (FIXED): Create NAS Controller
import os

# Create directory
os.makedirs('src/nas', exist_ok=True)

with open('src/nas/__init__.py', 'w') as f:
    f.write('# NAS modules\n')

print("✓ Created src/nas/__init__.py")

nas_code = '''import random
import numpy as np
from typing import List, Dict, Tuple, Any
import torch
from dataclasses import asdict
import json
from pathlib import Path

class EvolutionaryNAS:
    """
    Evolutionary Algorithm based NAS for ViT architecture and compression search.
    """

    def __init__(self, search_space, supernet, device='cpu'):
        """
        Initialize NAS controller.

        Args:
            search_space: SearchSpace instance
            supernet: ViTSupernet model
            device: 'cpu' or 'cuda'
        """
        self.search_space = search_space
        self.supernet = supernet
        self.device = device
        self.supernet.to(device)

        self.population = []
        self.fitness_history = []
        self.pareto_front = []

        print(f"✓ EvolutionaryNAS initialized on {device}")

    def random_population(self, pop_size: int) -> List[Dict]:
        """Generate random population of configs."""
        population = []
        for _ in range(pop_size):
            config = self.search_space.random_sample()
            is_valid, _ = self.search_space.validate_config(config)

            # Keep sampling until we get a valid config
            while not is_valid:
                config = self.search_space.random_sample()
                is_valid, _ = self.search_space.validate_config(config)

            population.append({
                'config': config,
                'accuracy': 0.0,
                'latency': 0.0,
                'model_size': 0.0,
                'fitness': 0.0,
            })

        return population

    def evaluate_config(self, config, dummy_input=None) -> Dict[str, float]:
        """
        Evaluate a configuration.

        For now, this is a mock evaluation (random scores).
        In real training, this would involve:
        - Training the subnet
        - Measuring accuracy on validation set
        - Profiling latency on device
        - Estimating model size

        Args:
            config: SearchConfig
            dummy_input: Dummy input for forward pass

        Returns:
            Dictionary with metrics
        """
        if dummy_input is None:
            dummy_input = torch.randn(1, 3, 224, 224).to(self.device)

        # Forward pass to ensure config is compatible
        with torch.no_grad():
            output = self.supernet(dummy_input, config=config)

        # Mock metrics (in real training, these would be actual measured values)
        accuracy = random.uniform(0.75, 0.95)
        latency = 50 + config.architecture.depth * 5  # Crude estimation
        model_size = 100 + config.architecture.embed_dim * 0.01

        return {
            'accuracy': accuracy,
            'latency': latency,
            'model_size': model_size,
        }

    def compute_fitness(self, metrics: Dict[str, float],
                       target_latency: float = 100.0) -> float:
        """
        Compute fitness score (multi-objective).

        Fitness = Accuracy - λ1 * (Latency - Target) - λ2 * Model_Size

        Args:
            metrics: Dictionary with accuracy, latency, model_size
            target_latency: Target latency in ms

        Returns:
            Fitness score
        """
        accuracy_weight = 1.0
        latency_weight = 0.01
        model_size_weight = 0.001

        accuracy_term = accuracy_weight * metrics['accuracy']
        latency_term = latency_weight * max(0, metrics['latency'] - target_latency)
        model_size_term = model_size_weight * metrics['model_size']

        fitness = accuracy_term - latency_term - model_size_term

        return fitness

    def evaluate_population(self, population: List[Dict],
                           dummy_input=None) -> List[Dict]:
        """Evaluate all configs in population."""
        for individual in population:
            metrics = self.evaluate_config(individual['config'], dummy_input)
            individual['accuracy'] = metrics['accuracy']
            individual['latency'] = metrics['latency']
            individual['model_size'] = metrics['model_size']
            individual['fitness'] = self.compute_fitness(metrics)

        return population

    def selection(self, population: List[Dict], k: int = 2) -> List[Dict]:
        """Tournament selection."""
        selected = []
        for _ in range(len(population)):
            tournament = random.sample(population, k)
            winner = max(tournament, key=lambda x: x['fitness'])
            selected.append(winner)

        return selected

    def crossover(self, parent1_config, parent2_config):
        """Simple crossover: mix architecture from both parents."""
        child_config = self.search_space.random_sample()

        # 50% chance to inherit from each parent
        if random.random() > 0.5:
            child_config.architecture.depth = parent1_config.architecture.depth
        else:
            child_config.architecture.depth = parent2_config.architecture.depth

        return child_config

    def mutation(self, config):
        """Mutate a configuration."""
        # Random mutation: change one aspect
        mutation_type = random.choice(['depth', 'pruning', 'quantization'])

        if mutation_type == 'depth':
            config.architecture.depth = random.choice([6, 9, 12, 16])
        elif mutation_type == 'pruning':
            config.compression.pruning.enabled = not config.compression.pruning.enabled
        elif mutation_type == 'quantization':
            config.compression.quantization.enabled = not config.compression.quantization.enabled

        return config

    def search(self, num_generations: int = 10, pop_size: int = 20,
               mutation_rate: float = 0.2) -> Dict[str, Any]:
        """
        Run evolutionary search.

        Args:
            num_generations: Number of generations
            pop_size: Population size
            mutation_rate: Probability of mutation

        Returns:
            Best configs and search history
        """
        print(f"\\nStarting Evolutionary NAS Search...")
        print(f"  Generations: {num_generations}")
        print(f"  Population: {pop_size}")
        print(f"  Mutation rate: {mutation_rate}")
        print()

        # Create dummy input
        dummy_input = torch.randn(1, 3, 224, 224).to(self.device)

        # Initialize population
        population = self.random_population(pop_size)

        for gen in range(num_generations):
            print(f"Generation {gen+1}/{num_generations}")

            # Evaluate
            population = self.evaluate_population(population, dummy_input)

            # Get best individual
            best = max(population, key=lambda x: x['fitness'])
            self.fitness_history.append(best['fitness'])

            print(f"  Best fitness: {best['fitness']:.4f}")
            print(f"  Accuracy: {best['accuracy']:.4f}")
            print(f"  Latency: {best['latency']:.1f}ms")
            print(f"  Model size: {best['model_size']:.1f}MB")

            # Update Pareto front
            self.pareto_front = self._compute_pareto_front(population)

            # Selection
            selected = self.selection(population, k=2)

            # Create next generation
            new_population = []
            for i in range(pop_size):
                # Crossover
                p1 = random.choice(selected)
                p2 = random.choice(selected)
                child_config = self.crossover(p1['config'], p2['config'])

                # Mutation
                if random.random() < mutation_rate:
                    child_config = self.mutation(child_config)

                is_valid, _ = self.search_space.validate_config(child_config)
                if not is_valid:
                    child_config = self.search_space.random_sample()

                new_population.append({
                    'config': child_config,
                    'accuracy': 0.0,
                    'latency': 0.0,
                    'model_size': 0.0,
                    'fitness': 0.0,
                })

            population = new_population
            print()

        # Return results
        best_individual = max(self.pareto_front, key=lambda x: x['fitness'])

        return {
            'best_config': best_individual['config'],
            'best_fitness': best_individual['fitness'],
            'best_accuracy': best_individual['accuracy'],
            'best_latency': best_individual['latency'],
            'pareto_front': self.pareto_front,
            'fitness_history': self.fitness_history,
        }

    def _compute_pareto_front(self, population: List[Dict]) -> List[Dict]:
        """Compute Pareto front from population."""
        pareto = []

        for individual in population:
            is_dominated = False
            for other in population:
                if other is individual:
                    continue

                # Check if other dominates individual
                # (higher fitness, lower latency, lower model size)
                if (other['fitness'] > individual['fitness'] and
                    other['latency'] < individual['latency']):
                    is_dominated = True
                    break

            if not is_dominated:
                pareto.append(individual)

        return pareto
'''

with open('src/nas/controller.py', 'w') as f:
    f.write(nas_code)

print("✓ Created src/nas/controller.py")
print("\n✅ NAS Controller created!")

In [ ]:
# Cell 10: Test NAS Controller
import torch
from src.supernet.vit_supernet import ViTSupernet
from src.search_space.search_space import SearchSpace
from src.nas.controller import EvolutionaryNAS

print("="*70)
print("NAS CONTROLLER TEST")
print("="*70)

# Create components
search_space = SearchSpace("config/search_space.yaml")
supernet = ViTSupernet(img_size=224, patch_size=16, num_classes=1000)
nas = EvolutionaryNAS(search_space, supernet, device='cpu')

print(f"\n✓ NAS Controller initialized")

# Run small search
results = nas.search(num_generations=3, pop_size=5, mutation_rate=0.2)

print("="*70)
print("SEARCH RESULTS")
print("="*70)
print(f"\nBest Configuration Found:")
print(f"  Fitness: {results['best_fitness']:.4f}")
print(f"  Accuracy: {results['best_accuracy']:.4f}")
print(f"  Latency: {results['best_latency']:.1f}ms")
print(f"  Pareto front size: {len(results['pareto_front'])}")

print(f"\n✅ NAS Controller Test PASSED!")
print("="*70)

In [12]:
# Cell 16: Create Training & Evaluation Pipeline
import os

os.makedirs('src/training', exist_ok=True)

with open('src/training/__init__.py', 'w') as f:
    f.write('# Training modules\n')

training_code = '''import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import time
from typing import Dict, Tuple
import numpy as np

class ProxyAccuracyEvaluator:
    """
    Evaluates model configurations using proxy accuracy on CIFAR-10.

    Instead of full ImageNet training (weeks), we use CIFAR-10 (1 epoch = minutes).
    This gives REAL signal for NAS without massive computation.
    """

    def __init__(self, device='cpu', dataset_size=5000):
        """
        Args:
            device: 'cpu' or 'cuda'
            dataset_size: Number of CIFAR-10 samples to use (full=50k)
        """
        self.device = device
        self.dataset_size = dataset_size

        # Load CIFAR-10
        print(f"Loading CIFAR-10 dataset ({dataset_size} samples)...")

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465),
                                (0.2023, 0.1994, 0.2010))
        ])

        # Training set for evaluation
        full_train = datasets.CIFAR10(
            root='./data/cifar1', train=True, download=True, transform=transform
        )
        self.train_data = Subset(full_train, range(min(dataset_size, len(full_train))))

        # Test set for validation
        full_test = datasets.CIFAR10(
            root='./data/cifar1', train=False, download=True, transform=transform
        )
        self.val_data = full_test

        self.train_loader = DataLoader(self.train_data, batch_size=64, shuffle=True)
        self.val_loader = DataLoader(self.val_data, batch_size=64, shuffle=False)

        print(f"✓ Loaded {len(self.train_data)} training samples")
        print(f"✓ Loaded {len(self.val_data)} validation samples")

    def train_epoch(self, model, learning_rate=0.01) -> float:
        """
        Train model for 1 epoch on CIFAR-10 training set.

        Args:
            model: Model to train
            learning_rate: Learning rate

        Returns:
            Average training loss
        """
        model.train()
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
        criterion = nn.CrossEntropyLoss()

        total_loss = 0.0
        num_batches = 0

        for images, labels in self.train_loader:
            images, labels = images.to(self.device), labels.to(self.device)

            # Resize images from 32x32 to 224x224 for ViT
            images = F.interpolate(images, size=224, mode='bilinear', align_corners=False)

            # Forward pass
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        return avg_loss

    def evaluate(self, model) -> float:
        """
        Evaluate model accuracy on CIFAR-10 validation set.

        Args:
            model: Model to evaluate

        Returns:
            Accuracy (0-1)
        """
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in self.val_loader:
                images, labels = images.to(self.device), labels.to(self.device)

                # Resize images
                images = F.interpolate(images, size=224, mode='bilinear', align_corners=False)

                # Forward pass
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        return accuracy

    def measure_latency(self, model, num_iterations=100) -> float:
        """
        Measure inference latency on random input.

        Args:
            model: Model to profile
            num_iterations: Number of forward passes for averaging

        Returns:
            Average latency in milliseconds
        """
        model.eval()
        dummy_input = torch.randn(1, 3, 224, 224).to(self.device)

        # Warmup
        with torch.no_grad():
            for _ in range(10):
                _ = model(dummy_input)

        # Measure
        if self.device == 'cuda':
            torch.cuda.synchronize()

        start_time = time.time()

        with torch.no_grad():
            for _ in range(num_iterations):
                _ = model(dummy_input)

        if self.device == 'cuda':
            torch.cuda.synchronize()

        end_time = time.time()

        total_time_ms = (end_time - start_time) * 1000
        avg_latency = total_time_ms / num_iterations

        return avg_latency

    def evaluate_config(self, model, config=None) -> Dict[str, float]:
        """
        Full evaluation: train 1 epoch, then evaluate.

        This is the REAL evaluation signal for NAS!

        Args:
            model: Model to evaluate
            config: SearchConfig (for reference)

        Returns:
            Dictionary with real metrics
        """
        try:
            print(f"  Training 1 epoch...", end="", flush=True)
            train_loss = self.train_epoch(model, learning_rate=0.01)
            print(f" loss={train_loss:.4f}")

            print(f"  Evaluating accuracy...", end="", flush=True)
            accuracy = self.evaluate(model)
            print(f" acc={accuracy:.4f}")

            print(f"  Measuring latency...", end="", flush=True)
            latency = self.measure_latency(model, num_iterations=50)
            print(f" {latency:.2f}ms")

            # Estimate model size
            num_params = sum(p.numel() for p in model.parameters())
            model_size_mb = (num_params * 4) / (1024 * 1024)  # 4 bytes per float32

            return {
                'accuracy': accuracy,
                'latency': latency,
                'model_size': model_size_mb,
                'train_loss': train_loss,
            }

        except Exception as e:
            print(f" ERROR: {e}")
            # Return worst-case metrics if evaluation fails
            return {
                'accuracy': 0.1,
                'latency': 500.0,
                'model_size': 500.0,
                'train_loss': 10.0,
            }
'''

with open('src/training/evaluator.py', 'w') as f:
    f.write(training_code)

print("✓ Created src/training/evaluator.py")
print("\n✅ Real evaluation pipeline created!")

✓ Created src/training/evaluator.py

✅ Real evaluation pipeline created!


In [13]:
# Cell 17: Update NAS to use real evaluation
nas_update_code = '''
# Add this to EvolutionaryNAS class in src/nas/controller.py

def set_real_evaluator(self, evaluator):
    """
    Set the real evaluator (replaces mock evaluation).

    Args:
        evaluator: ProxyAccuracyEvaluator instance
    """
    self.evaluator = evaluator
    print("✓ Real evaluator set")

def evaluate_config_real(self, model, config):
    """
    Evaluate using REAL proxy accuracy (CIFAR-10).

    This replaces the random mock evaluation!

    Args:
        model: ViT model
        config: SearchConfig specifying which layers to use

    Returns:
        Dictionary with real metrics
    """
    # Create subnet from config
    model_copy = model  # In practice, you'd create a subnet

    # Use real evaluator
    metrics = self.evaluator.evaluate_config(model_copy, config)

    return metrics
'''

# Read current controller and add the update
with open('src/nas/controller.py', 'r') as f:
    controller_content = f.read()

# Add real evaluation method to EvolutionaryNAS class
if 'set_real_evaluator' not in controller_content:
    # Find the class definition and add the method
    insertion_point = controller_content.find('def random_population')

    new_method = '''
    def set_real_evaluator(self, evaluator):
        """Set real evaluator (replaces mock evaluation)."""
        self.evaluator = evaluator
        self.use_real_evaluation = True
        print("✓ Real evaluator set - NAS will now optimize REAL metrics!")

    def evaluate_config(self, config, dummy_input=None) -> Dict[str, float]:
        """
        Evaluate configuration.

        NOW USES REAL ACCURACY IF EVALUATOR IS SET!
        Falls back to mock if no evaluator.
        """
        if hasattr(self, 'evaluator') and self.use_real_evaluation:
            # Use REAL evaluation
            metrics = self.evaluator.evaluate_config(self.supernet, config)
        else:
            # Use mock evaluation (old code)
            if dummy_input is None:
                dummy_input = torch.randn(1, 3, 224, 224).to(self.device)

            try:
                with torch.no_grad():
                    output = self.supernet(dummy_input, config=config)
            except Exception as e:
                return {
                    'accuracy': 0.5,
                    'latency': 500.0,
                    'model_size': 500.0,
                }

            # Mock metrics
            base_accuracy = 0.75
            depth_bonus = (config.architecture.depth - 6) * 0.01
            embed_bonus = (config.architecture.embed_dim - 192) * 0.0002
            pruning_penalty = config.compression.pruning.layer_wise_ratios[0] * 0.1
            quant_penalty = (16 - config.compression.quantization.layer_wise_bits[0]) * 0.005

            accuracy = min(0.95, max(0.5,
                base_accuracy + depth_bonus + embed_bonus - pruning_penalty - quant_penalty
            ))

            latency = 30 + config.architecture.depth * 8 + (config.architecture.embed_dim - 192) * 0.05

            if config.compression.pruning.enabled:
                latency *= (1 - config.compression.pruning.layer_wise_ratios[0] * 0.5)
            if config.compression.quantization.enabled:
                latency *= (config.compression.quantization.layer_wise_bits[0] / 8.0)

            base_size = 50
            model_size = base_size + config.architecture.depth * 5 + (config.architecture.embed_dim - 192) * 0.02

            if config.compression.pruning.enabled:
                model_size *= (1 - config.compression.pruning.layer_wise_ratios[0])
            if config.compression.quantization.enabled:
                model_size *= (config.compression.quantization.layer_wise_bits[0] / 8.0)

            metrics = {
                'accuracy': accuracy,
                'latency': latency,
                'model_size': model_size,
            }

        return metrics

    '''

    # Update the file with new methods
    updated_content = controller_content[:insertion_point] + new_method + controller_content[insertion_point:]

    with open('src/nas/controller.py', 'w') as f:
        f.write(updated_content)

    print("✓ Updated EvolutionaryNAS with real evaluation support")
else:
    print("✓ Real evaluation methods already present")

print("\n✅ NAS Controller updated!")

✓ Updated EvolutionaryNAS with real evaluation support

✅ NAS Controller updated!


In [15]:
# Cell 17 (FIXED): Properly update NAS Controller
import os

# Read the current controller
with open('src/nas/controller.py', 'r') as f:
    controller_content = f.read()

# Find where to insert the new methods
# We'll add them right after the __init__ method

# First, let's create a new updated version of the file
new_nas_code = '''import random
import numpy as np
from typing import List, Dict, Tuple, Any
import torch
from dataclasses import asdict
import json
from pathlib import Path

class EvolutionaryNAS:
    """
    Evolutionary Algorithm based NAS for ViT architecture and compression search.

    This class implements a genetic algorithm that:
    1. Maintains a population of ViT configurations
    2. Evaluates each configuration's fitness
    3. Selects the best configurations
    4. Creates new configurations through crossover and mutation
    5. Repeats for multiple generations to find optimal architectures
    """

    def __init__(self, search_space, supernet, device='cpu'):
        """
        Initialize NAS controller.

        Args:
            search_space: SearchSpace instance
            supernet: ViTSupernet model
            device: 'cpu' or 'cuda'
        """
        self.search_space = search_space
        self.supernet = supernet
        self.device = device
        self.supernet.to(device)

        self.population = []
        self.fitness_history = []
        self.pareto_front = []
        self.generation_stats = []

        # Real evaluation support
        self.evaluator = None
        self.use_real_evaluation = False

        print(f"✓ EvolutionaryNAS initialized on {device}")

    def set_real_evaluator(self, evaluator):
        """
        Set the real evaluator (replaces mock evaluation with CIFAR-10 training).

        Args:
            evaluator: ProxyAccuracyEvaluator instance
        """
        self.evaluator = evaluator
        self.use_real_evaluation = True
        print("✓ Real evaluator set - NAS will now optimize REAL metrics!")

    def random_population(self, pop_size: int) -> List[Dict]:
        """
        Generate random population of configs.

        Args:
            pop_size: Number of configs to generate

        Returns:
            List of individuals (each with config and metrics)
        """
        population = []
        for _ in range(pop_size):
            config = self.search_space.random_sample()
            is_valid, _ = self.search_space.validate_config(config)

            # Keep sampling until we get a valid config
            attempts = 0
            while not is_valid and attempts < 10:
                config = self.search_space.random_sample()
                is_valid, _ = self.search_space.validate_config(config)
                attempts += 1

            if is_valid:
                population.append({
                    'config': config,
                    'accuracy': 0.0,
                    'latency': 0.0,
                    'model_size': 0.0,
                    'fitness': 0.0,
                })

        return population

    def evaluate_config(self, config, dummy_input=None) -> Dict[str, float]:
        """
        Evaluate a configuration.

        If real_evaluator is set: uses CIFAR-10 training for real accuracy
        Otherwise: uses mock evaluation (random scores)

        Args:
            config: SearchConfig
            dummy_input: Dummy input for forward pass

        Returns:
            Dictionary with metrics
        """
        if self.use_real_evaluation and self.evaluator is not None:
            # Use REAL evaluation with CIFAR-10
            print(f"\\n  [REAL] Evaluating config (depth={config.architecture.depth}, "
                  f"embed_dim={config.architecture.embed_dim})...")
            metrics = self.evaluator.evaluate_config(self.supernet, config)
            return metrics
        else:
            # Use mock evaluation (old approach)
            if dummy_input is None:
                dummy_input = torch.randn(1, 3, 224, 224).to(self.device)

            # Forward pass to ensure config is compatible
            try:
                with torch.no_grad():
                    output = self.supernet(dummy_input, config=config)
            except Exception as e:
                return {
                    'accuracy': 0.5,
                    'latency': 500.0,
                    'model_size': 500.0,
                }

            # Mock metrics (estimated)
            accuracy = random.uniform(0.75, 0.95)
            latency = 50 + config.architecture.depth * 5
            model_size = 100 + config.architecture.embed_dim * 0.01

            return {
                'accuracy': accuracy,
                'latency': latency,
                'model_size': model_size,
            }

    def compute_fitness(self, metrics: Dict[str, float],
                       target_latency: float = 100.0,
                       target_model_size: float = 50.0) -> float:
        """
        Compute fitness score (multi-objective optimization).

        Fitness = w1 * Accuracy - w2 * |Latency - Target| - w3 * |ModelSize - Target|

        Higher is better!

        Args:
            metrics: Dictionary with accuracy, latency, model_size
            target_latency: Target latency in ms
            target_model_size: Target model size in MB

        Returns:
            Fitness score (higher = better)
        """
        # Weights for multi-objective optimization
        accuracy_weight = 1.0      # Accuracy is most important
        latency_weight = 0.005     # Latency penalty
        model_size_weight = 0.001  # Model size penalty

        # Compute fitness terms
        accuracy_term = accuracy_weight * metrics['accuracy']

        # Penalize latency if it exceeds target
        latency_diff = max(0, metrics['latency'] - target_latency)
        latency_term = latency_weight * latency_diff

        # Penalize model size if it exceeds target
        model_size_diff = max(0, metrics['model_size'] - target_model_size)
        model_size_term = model_size_weight * model_size_diff

        fitness = accuracy_term - latency_term - model_size_term

        return fitness

    def evaluate_population(self, population: List[Dict],
                           dummy_input=None) -> List[Dict]:
        """
        Evaluate all configs in population.

        Args:
            population: List of individuals
            dummy_input: Dummy input tensor

        Returns:
            Population with updated fitness scores
        """
        for i, individual in enumerate(population):
            print(f"\\n  Individual {i+1}/{len(population)}:")
            metrics = self.evaluate_config(individual['config'], dummy_input)
            individual['accuracy'] = metrics['accuracy']
            individual['latency'] = metrics['latency']
            individual['model_size'] = metrics['model_size']
            individual['fitness'] = self.compute_fitness(metrics)

        return population

    def selection(self, population: List[Dict], k: int = 2) -> List[Dict]:
        """
        Tournament selection: pick k random individuals, return best one.
        Repeat to create new population.

        Args:
            population: List of individuals
            k: Tournament size

        Returns:
            Selected population (same size)
        """
        selected = []
        for _ in range(len(population)):
            tournament = random.sample(population, k)
            winner = max(tournament, key=lambda x: x['fitness'])
            selected.append(winner)

        return selected

    def crossover(self, parent1_config, parent2_config):
        """
        Create a child config by mixing traits from both parents.

        Args:
            parent1_config: First parent config
            parent2_config: Second parent config

        Returns:
            Child config
        """
        from copy import deepcopy

        child_config = deepcopy(parent1_config)

        # Mix architecture parameters
        if random.random() > 0.5:
            child_config.architecture.depth = parent2_config.architecture.depth
        if random.random() > 0.5:
            child_config.architecture.embed_dim = parent2_config.architecture.embed_dim
        if random.random() > 0.5:
            child_config.architecture.num_heads = parent2_config.architecture.num_heads
        if random.random() > 0.5:
            child_config.architecture.mlp_ratio = parent2_config.architecture.mlp_ratio

        return child_config

    def mutation(self, config):
        """
        Randomly mutate a configuration to introduce variation.

        Args:
            config: Configuration to mutate

        Returns:
            Mutated configuration
        """
        from copy import deepcopy
        mutation_rate = 0.1

        config = deepcopy(config)

        # Mutate architecture
        if random.random() < mutation_rate:
            config.architecture.depth = random.choice([6, 9, 12, 16])
        if random.random() < mutation_rate:
            config.architecture.embed_dim = random.choice([192, 384, 576, 768])

        return config

    def search(self, num_generations: int = 10, pop_size: int = 20,
               mutation_rate: float = 0.2,
               target_latency: float = 100.0,
               target_model_size: float = 50.0) -> Dict[str, Any]:
        """
        Run evolutionary search algorithm.

        Args:
            num_generations: Number of generations to evolve
            pop_size: Population size per generation
            mutation_rate: Probability of mutation
            target_latency: Target latency in ms
            target_model_size: Target model size in MB

        Returns:
            Dictionary with best configs and search history
        """
        print(f"\\n" + "="*70)
        print("STARTING EVOLUTIONARY NAS SEARCH")
        print("="*70)
        print(f"  Generations: {num_generations}")
        print(f"  Population size: {pop_size}")
        print(f"  Real evaluation: {self.use_real_evaluation}")
        print("="*70)

        # Create dummy input
        dummy_input = torch.randn(1, 3, 224, 224).to(self.device)

        # Initialize population
        print(f"\\nGenerating initial population ({pop_size} configs)...")
        population = self.random_population(pop_size)
        print(f"✓ Generated {len(population)} configurations")

        for gen in range(num_generations):
            print(f"\\n" + "="*70)
            print(f"GENERATION {gen+1}/{num_generations}")
            print("="*70)

            # Evaluate all individuals
            population = self.evaluate_population(population, dummy_input)

            # Get statistics
            fitnesses = [ind['fitness'] for ind in population]
            accuracies = [ind['accuracy'] for ind in population]
            latencies = [ind['latency'] for ind in population]

            best = max(population, key=lambda x: x['fitness'])
            avg_fitness = np.mean(fitnesses)

            self.fitness_history.append(best['fitness'])

            stats = {
                'generation': gen,
                'best_fitness': best['fitness'],
                'avg_fitness': avg_fitness,
                'best_accuracy': best['accuracy'],
                'best_latency': best['latency'],
            }
            self.generation_stats.append(stats)

            print(f"\\nGeneration {gen+1} Summary:")
            print(f"  Best fitness:    {best['fitness']:.6f}")
            print(f"  Avg fitness:     {avg_fitness:.6f}")
            print(f"  Best accuracy:   {best['accuracy']:.4f}")
            print(f"  Best latency:    {best['latency']:.2f}ms")
            print(f"  Best depth:      {best['config'].architecture.depth}")
            print(f"  Best embed_dim:  {best['config'].architecture.embed_dim}")

            # Update Pareto front
            self.pareto_front = self._compute_pareto_front(population)
            print(f"  Pareto front:    {len(self.pareto_front)} configs")

            # Selection
            selected = self.selection(population, k=2)

            # Create next generation
            new_population = []
            for i in range(pop_size):
                # Crossover
                p1 = random.choice(selected)
                p2 = random.choice(selected)
                child_config = self.crossover(p1['config'], p2['config'])

                # Mutation
                if random.random() < mutation_rate:
                    child_config = self.mutation(child_config)

                # Validate
                is_valid, _ = self.search_space.validate_config(child_config)
                if not is_valid:
                    child_config = self.search_space.random_sample()

                new_population.append({
                    'config': child_config,
                    'accuracy': 0.0,
                    'latency': 0.0,
                    'model_size': 0.0,
                    'fitness': 0.0,
                })

            population = new_population

        # Final evaluation
        print(f"\\nFinal evaluation...")
        population = self.evaluate_population(population, dummy_input)
        self.pareto_front = self._compute_pareto_front(population)

        best_individual = max(self.pareto_front, key=lambda x: x['fitness'])

        print(f"\\n" + "="*70)
        print("SEARCH COMPLETED")
        print("="*70)
        print(f"Final Pareto front size: {len(self.pareto_front)}")
        print(f"Best individual accuracy: {best_individual['accuracy']:.4f}")
        print(f"Best individual latency: {best_individual['latency']:.2f}ms")
        print("="*70 + "\\n")

        return {
            'best_config': best_individual['config'],
            'best_fitness': best_individual['fitness'],
            'best_accuracy': best_individual['accuracy'],
            'best_latency': best_individual['latency'],
            'best_model_size': best_individual['model_size'],
            'pareto_front': self.pareto_front,
            'fitness_history': self.fitness_history,
            'generation_stats': self.generation_stats,
        }

    def _compute_pareto_front(self, population: List[Dict]) -> List[Dict]:
        """
        Compute Pareto front: non-dominated solutions.

        Args:
            population: List of individuals

        Returns:
            Pareto front (subset of population)
        """
        pareto = []

        for individual in population:
            is_dominated = False
            for other in population:
                if other is individual:
                    continue

                # Check if other dominates individual
                if (other['fitness'] > individual['fitness'] and
                    other['latency'] < individual['latency']):
                    is_dominated = True
                    break

            if not is_dominated:
                pareto.append(individual)

        return sorted(pareto, key=lambda x: x['fitness'], reverse=True)
'''

# Write the complete updated file
with open('src/nas/controller.py', 'w') as f:
    f.write(new_nas_code)

print("✓ Completely rewrote src/nas/controller.py")
print("✓ Added set_real_evaluator() method")
print("✓ Added real evaluation support to evaluate_config()")
print("\n✅ NAS Controller properly updated!")

✓ Completely rewrote src/nas/controller.py
✓ Added set_real_evaluator() method
✓ Added real evaluation support to evaluate_config()

✅ NAS Controller properly updated!
